In [ ]:
import torch
import gc
import matplotlib.pyplot as plt
from transformers import AutoModelForCausalLM, AutoTokenizer, AutoConfig
from dataset import get_dataloader
from trainer import train_custom

print("Initializing...")
model_id = "PleIAs/Baguettotron"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

tokenizer = AutoTokenizer.from_pretrained(model_id)
if tokenizer.pad_token is None:
    tokenizer.add_special_tokens({'pad_token': '[PAD]'})

config = AutoConfig.from_pretrained(model_id)
dl_synth = get_dataloader("./data/synth_001.parquet", tokenizer, 4, 256)
random_batch = next(iter(dl_synth))

print("Loading Model A...")
model_a = AutoModelForCausalLM.from_config(config)
model_a.resize_token_embeddings(len(tokenizer))
model_a = model_a.to(torch.bfloat16).to(device)
model_a.gradient_checkpointing_enable()

opt_a = torch.optim.AdamW(model_a.parameters(), lr=1e-4, weight_decay=0.01)

print("\n--- PHASE 1: Model A Overfitting (3 minutes) ---")
train_custom(model_a, dl_synth, opt_a, device, target_time=20, single_batch=random_batch, accumulation_steps=8)

print("\n--- PHASE 2: Model A Main Training (5 minutes) ---")
loss_a_synth = train_custom(model_a, dl_synth, opt_a, device, target_time=180, accumulation_steps=8)

print("\nCleaning up memory...")
del model_a
del opt_a
torch.cuda.empty_cache()
gc.collect()

print("Loading Model B...")
model_b = AutoModelForCausalLM.from_config(config)
model_b.resize_token_embeddings(len(tokenizer))
model_b = model_b.to(torch.bfloat16).to(device)
model_b.gradient_checkpointing_enable()

opt_b = torch.optim.AdamW(model_b.parameters(), lr=1e-4, weight_decay=0.01)

print("\n--- PHASE 3: Model B Main Training (8 minutes) ---")
loss_b_synth = train_custom(model_b, dl_synth, opt_b, device, target_time=200, accumulation_steps=8)

print("\nPlotting results...")
plt.plot(loss_a_synth, label='Model A (After 3m Overfit)')
plt.plot(loss_b_synth, label='Model B (8m Standard)')
plt.xlabel('Iterations')
plt.ylabel('Loss')
plt.legend()
plt.show()

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

def smooth_curve(data, window=50):
    if len(data) < window: return data
    return np.convolve(data, np.ones(window)/window, mode='valid')

offset = 59
window_size = 30 

smooth_a = smooth_curve(loss_a_synth, window=window_size)
smooth_b = smooth_curve(loss_b_synth, window=window_size)

plt.figure(figsize=(12, 7), dpi=100)
plt.style.use('seaborn-v0_8-whitegrid')

x_b = np.arange(len(smooth_b))
plt.plot(x_b, smooth_b, label='Model B (Standard: Random Start)', color='#2980b9', linewidth=2.5)

x_a = np.arange(offset, offset + len(smooth_a))
plt.plot(x_a, smooth_a, label='Model A', color='#e67e22', linewidth=2.5)

plt.axvline(x=offset, color='grey', linestyle='--', alpha=0.5)
#plt.text(offset + 10, plt.ylim()[1]*0.95, 'Main Training Starts', color='grey', fontsize=10)
plt.text(offset + 5, 8.0, 'Main Training Starts', color='grey', rotation=90, verticalalignment='bottom')

plt.title('Honest Compute-Budget Comparison: Overfit Init vs Standard', fontsize=16, fontweight='bold', pad=20)
plt.xlabel('Total Computational Steps (Iterations)', fontsize=13)
plt.ylabel('Cross-Entropy Loss (Smoothed)', fontsize=13)

plt.annotate(f'Start Loss: {loss_b_synth[0]:.2f}', 
             xy=(0, smooth_b[0]), 
             xytext=(20, smooth_b[0] + 0.4),
             arrowprops=dict(arrowstyle='->', color='black', lw=1),
             fontsize=10)



plt.legend(fontsize=12, frameon=True, shadow=True)
plt.grid(True, linestyle='--', alpha=0.7)

plt.tight_layout()
plt.savefig("llm_small.png")
plt.show()